In [1]:
import numpy as np
import pandas as pd
from sklearn.ensemble import IsolationForest

### DATA COLLECTION

In [2]:

def generate_server_data():
    np.random.seed(42)

    timestamps = pd.date_range(
        start="2026-10-01 10:00:00",
        periods=500,
        freq="5min"
    )

    servers = ["Server01", "Server02"]
    data = []

    for ts in timestamps:
        for server in servers:

            # Generate normal operating values
            cpu = np.random.normal(55, 10)
            memory = np.random.normal(65, 8)
            network = np.random.normal(350, 50)
            response_time = np.random.normal(150, 25)

            # Introduce occasional artificial anomalies
            if np.random.rand() < 0.03:
                cpu = np.random.uniform(92, 99)
                memory = np.random.uniform(90, 98)
                response_time = np.random.uniform(3500, 5500)

            # IMPORTANT: Keep append INSIDE both loops
            data.append({
                "Timestamp": ts,
                "Server": server,
                "CPU": round(max(0, min(100, cpu)), 2),
                "Memory": round(max(0, min(100, memory)), 2),
                "Network": round(max(0, network), 2),
                "Response_Time": round(max(0, response_time), 2)
            })

    # Create the DataFrame AFTER both loops finish
    df = pd.DataFrame(data)

    return df


In [3]:
df = generate_server_data()
print("--- Raw Dataset Sample ---")
print(df.head())

--- Raw Dataset Sample ---
            Timestamp    Server    CPU  Memory  Network  Response_Time
0 2026-10-01 10:00:00  Server01  59.97   63.89   382.38         188.08
1 2026-10-01 10:00:00  Server02  57.79   73.08   320.96         136.87
2 2026-10-01 10:05:00  Server01  57.42   49.69   263.75         135.94
3 2026-10-01 10:05:00  Server02  50.72   59.06   314.83          96.51
4 2026-10-01 10:10:00  Server01  55.68   53.60   322.78         152.77


### DATA CLEANING

In [4]:
df = df.drop_duplicates()
df = df.dropna()

# Remove any negative values or invalid metrics
df = df[(df['CPU'] >= 0) & (df['Memory'] >= 0) & (df['Response_Time'] >= 0)]

### FEATURE ENGINEERING


In [5]:
df['CPU_Load_Ratio'] = df['CPU'] / 100.0
df['Memory_Load_Ratio'] = df['Memory'] / 100.0
df['Requests_Per_Minute'] = (df['Network'] / 10).round(0)
df['Error_Rate'] = np.where(df['Response_Time'] > 2000, 1.0, 0.0)


### ANOMALY DETECTION MODEL (Isolation Forest)

In [6]:
# Selecting numerical features for the model
features = ['CPU', 'Memory', 'Network', 'Response_Time', 'Requests_Per_Minute']
X = df[features].copy()

In [7]:
# Check that all model inputs are numeric and valid
X = X.replace([np.inf, -np.inf], np.nan)

if X.isna().any().any():
    raise ValueError("Missing or invalid values found in model features.")


In [8]:
# Initialize and fit the Isolation Forest model (ideal for internships)[cite: 9, 10]
model = IsolationForest(n_estimators=100,contamination=0.03, random_state=42)
df["Prediction"] = model.fit_predict(X)
df["Anomaly_Score"] = model.decision_function(X)

In [9]:
# Isolation Forest outputs: 1 for normal, -1 for anomaly. Map them to readable text.
df["Status"] = df["Prediction"].map({1: "Normal",-1: "Anomaly"})

print(df["Status"].value_counts())

Status
Normal     970
Anomaly     30
Name: count, dtype: int64


In [10]:

print("Total monitoring rows:", len(df))
print("Unique servers:", df["Server"].nunique())
print("Rows per server:")
print(df["Server"].value_counts())


Total monitoring rows: 1000
Unique servers: 2
Rows per server:
Server
Server01    500
Server02    500
Name: count, dtype: int64


### ALERT SYSTEM MODULE

In [11]:
# Create a separate table containing detected anomalies
alerts = df[df["Prediction"] == -1].copy()

# Add alert status
alerts["Alert_Status"] = "ANOMALY DETECTED"

print("Model completed successfully!")
print("\nDetection summary:")
print(df["Status"].value_counts())

print("\nFirst five anomalies:")
display(alerts.head())


Model completed successfully!

Detection summary:
Status
Normal     970
Anomaly     30
Name: count, dtype: int64

First five anomalies:


,Timestamp,Server,CPU,Memory,Network,Response_Time,CPU_Load_Ratio,Memory_Load_Ratio,Requests_Per_Minute,Error_Rate,Prediction,Anomaly_Score,Status,Alert_Status
14,2026-10-01 10:35:00,Server01,92.76,90.25,219.01,4772.82,0.9276,0.9025,22.0,1.0,-1,-0.122676,Anomaly,ANOMALY DETECTED
19,2026-10-01 10:45:00,Server02,95.58,93.34,364.81,3944.22,0.9558,0.9334,36.0,1.0,-1,-0.048038,Anomaly,ANOMALY DETECTED
31,2026-10-01 11:15:00,Server02,92.71,95.31,361.31,3510.12,0.9271,0.9531,36.0,1.0,-1,-0.054730,Anomaly,ANOMALY DETECTED
127,2026-10-01 15:15:00,Server02,98.10,97.46,472.29,4630.27,0.9810,0.9746,47.0,1.0,-1,-0.130893,Anomaly,ANOMALY DETECTED
131,2026-10-01 15:25:00,Server02,94.33,93.19,403.55,4574.79,0.9433,0.9319,40.0,1.0,-1,-0.064486,Anomaly,ANOMALY DETECTED


In [12]:

print("Monitoring rows:", len(df))
print("Anomaly alert rows:", len(alerts))

display(df.head())
display(alerts.head())


Monitoring rows: 1000
Anomaly alert rows: 30


,Timestamp,Server,CPU,Memory,Network,Response_Time,CPU_Load_Ratio,Memory_Load_Ratio,Requests_Per_Minute,Error_Rate,Prediction,Anomaly_Score,Status
0,2026-10-01 10:00:00,Server01,59.97,63.89,382.38,188.08,0.5997,0.6389,38.0,0.0,1,0.223649,Normal
1,2026-10-01 10:00:00,Server02,57.79,73.08,320.96,136.87,0.5779,0.7308,32.0,0.0,1,0.223996,Normal
2,2026-10-01 10:05:00,Server01,57.42,49.69,263.75,135.94,0.5742,0.4969,26.0,0.0,1,0.090381,Normal
3,2026-10-01 10:05:00,Server02,50.72,59.06,314.83,96.51,0.5072,0.5906,31.0,0.0,1,0.166109,Normal
4,2026-10-01 10:10:00,Server01,55.68,53.60,322.78,152.77,0.5568,0.5360,32.0,0.0,1,0.199782,Normal


,Timestamp,Server,CPU,Memory,Network,Response_Time,CPU_Load_Ratio,Memory_Load_Ratio,Requests_Per_Minute,Error_Rate,Prediction,Anomaly_Score,Status,Alert_Status
14,2026-10-01 10:35:00,Server01,92.76,90.25,219.01,4772.82,0.9276,0.9025,22.0,1.0,-1,-0.122676,Anomaly,ANOMALY DETECTED
19,2026-10-01 10:45:00,Server02,95.58,93.34,364.81,3944.22,0.9558,0.9334,36.0,1.0,-1,-0.048038,Anomaly,ANOMALY DETECTED
31,2026-10-01 11:15:00,Server02,92.71,95.31,361.31,3510.12,0.9271,0.9531,36.0,1.0,-1,-0.054730,Anomaly,ANOMALY DETECTED
127,2026-10-01 15:15:00,Server02,98.10,97.46,472.29,4630.27,0.9810,0.9746,47.0,1.0,-1,-0.130893,Anomaly,ANOMALY DETECTED
131,2026-10-01 15:25:00,Server02,94.33,93.19,403.55,4574.79,0.9433,0.9319,40.0,1.0,-1,-0.064486,Anomaly,ANOMALY DETECTED


In [13]:

# STEP 6: EXPORT RESULTS FOR POWER BI

from pathlib import Path

OUTPUT_DIR = Path("output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

results_path = OUTPUT_DIR / "server_monitoring_results.csv"
alerts_path = OUTPUT_DIR / "server_anomaly_alerts.csv"

df.to_csv(results_path, index=False)
alerts.to_csv(alerts_path, index=False)

print("Monitoring results saved to:", results_path.resolve())
print("Anomaly alerts saved to:", alerts_path.resolve())


Monitoring results saved to: C:\Users\DELL\Downloads\OSMetrics\output\server_monitoring_results.csv
Anomaly alerts saved to: C:\Users\DELL\Downloads\OSMetrics\output\server_anomaly_alerts.csv
